# Importações

In [1]:
import pandas as pd
import os
import numpy as np
from sqlalchemy import text
from sqlalchemy import create_engine

PRPI_DATABASE_URL = os.getenv("PRPI_DATABASE_URL")
engine = create_engine(PRPI_DATABASE_URL)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

# Carregamento de dados e pré transformações

## Produções Técnicas

In [2]:
query = "select * from cnpq_producaotecnica order by tipo_pub asc;"
producoes_tecnicas_df = pd.read_sql(query, engine)

producoes_tecnicas_df = producoes_tecnicas_df.rename(columns={
    "id": "ID",
    "curriculo_id": "ID do Currículo",
    "sequencia": "Sequência",
    "titulo": "Título",
    "ano": "Ano",
    "pais": "País",
    "idioma": "Idioma",
    "flag_relevancia": "Relevância",
    "doi": "DOI",
    "tipo_pub": "Tipo de Publicação",
    "informacao_adicional_id": "ID de Informação Adicional"
})

### Padronização de Tipo de Publicação

In [3]:
producoes_tecnicas_df["Tipo de Publicação"] = (
    producoes_tecnicas_df["Tipo de Publicação"]
        .str.replace("-", " ", regex=False)
        .str.lower()
        .str.replace("cao", "ção", regex=False)
        .str.title()
)

## Membros de Projetos

In [72]:
query = """
  SELECT
    pr.id AS projeto_id,
    pr.titulo AS titulo_projeto,
    p.id as id_pessoa,
    p.nome AS nome_pessoa,
    p.email AS email_pessoa,
    (painelprpi_get_uo_info(pr.uo_id)).nome AS campus,
    CASE
        WHEN p.email ILIKE '%@aluno.ifce.edu.br' THEN 'Discente'
        WHEN s.eh_docente = TRUE THEN 'Docente'
        WHEN s.eh_tecnico_administrativo = TRUE THEN 'TAE'
        ELSE NULL
    END AS categoria_funcional,
    pp.vinculo,
    pp.ativo,
    pr.inicio_execucao,
    EXTRACT(YEAR FROM pr.inicio_execucao) AS ano_inicio_execucao,
    pr.area_conhecimento_id,
    ac.descricao AS descricao_area_conhecimento,
    pr.edital_id,
    pe.titulo AS titulo_edital,
    pe.tipo_edital,
    CASE pe.tipo_edital
        WHEN '1' THEN 'Pesquisa'
        WHEN '2' THEN 'Inovação'
        WHEN '3' THEN 'Pesquisa/Inovação - Contínuo'
        ELSE 'Desconhecido'
    END AS descricao_tipo_edital
    FROM pesquisa_projeto pr
    JOIN pesquisa_participacao pp ON pp.projeto_id = pr.id
    JOIN comum_vinculo cv ON cv.id = pp.vinculo_pessoa_id
    JOIN pessoa p ON p.id = cv.pessoa_id
    LEFT JOIN servidor s ON s.pessoa_fisica_id = p.id
    LEFT JOIN rh_areaconhecimento ac ON ac.id = pr.area_conhecimento_id
    LEFT JOIN pesquisa_edital pe ON pe.id = pr.edital_id
    LEFT JOIN edu_aluno alu ON alu.pessoa_fisica_id = p.id
    LEFT JOIN edu_situacaomatricula sitmat ON sitmat.id = alu.situacao_id
    ORDER BY pr.id, p.nome;
"""

with engine.connect() as conn:
    membros_projetos_df = pd.read_sql(text(query), conn)
    
membros_projetos_df = membros_projetos_df.rename(columns={
    "projeto_id": "ID do Projeto",
    "titulo_projeto": "Título do Projeto",
    "id_pessoa": "ID do Membro",
    "nome_pessoa": "Nome do Membro",
    "email_pessoa": "Email do Membro",
    "campus": "Campus",
    "categoria_funcional": "Categoria do Membro",
    "vinculo": "Vínculo com Projeto",
    "ativo": "Ativo",
    "inicio_execucao": "Início da Execução",
    "ano_inicio_execucao": "Ano de Início da Execução",
    "area_conhecimento_id": "ID da Área de Conhecimento",
    "descricao_area_conhecimento": "Área de Conhecimento",
    "edital_id": "ID do Edital",
    "titulo_edital": "Título do Edital",
    "tipo_edital": "ID do Tipo de Edital",
    "descricao_tipo_edital": "Tipo de Edital"
})

DatabaseError: Execution failed on sql '
  SELECT
    pr.id AS projeto_id,
    pr.titulo AS titulo_projeto,
    p.id as id_pessoa,
    p.nome AS nome_pessoa,
    p.email AS email_pessoa,
    (painelprpi_get_uo_info(pr.uo_id)).nome AS campus,
    CASE
        WHEN p.email ILIKE '%@aluno.ifce.edu.br' THEN 'Discente'
        WHEN s.eh_docente = TRUE THEN 'Docente'
        WHEN s.eh_tecnico_administrativo = TRUE THEN 'TAE'
        ELSE NULL
    END AS categoria_funcional,
    pp.vinculo,
    pp.ativo,
    pr.inicio_execucao,
    EXTRACT(YEAR FROM pr.inicio_execucao) AS ano_inicio_execucao,
    pr.area_conhecimento_id,
    ac.descricao AS descricao_area_conhecimento,
    pr.edital_id,
    pe.titulo AS titulo_edital,
    pe.tipo_edital,
    CASE pe.tipo_edital
        WHEN '1' THEN 'Pesquisa'
        WHEN '2' THEN 'Inovação'
        WHEN '3' THEN 'Pesquisa/Inovação - Contínuo'
        ELSE 'Desconhecido'
    END AS descricao_tipo_edital
    FROM pesquisa_projeto pr
    JOIN pesquisa_particip

### Padronização de Edital

In [ ]:
membros_projetos_df["Título do Edital"] = membros_projetos_df["Título do Edital"].str.upper()

conditions = [
    membros_projetos_df["Título do Edital"].str.startswith("PROGRAMA DE INICIAÇÃO CIENTÍFICA E TECNOLÓGICA VOLUNTÁRIA", na=False),
    membros_projetos_df["Título do Edital"].str.startswith("PIBITI", na=False),
    membros_projetos_df["Título do Edital"].str.startswith("PIBIC JR AF", na=False),
    membros_projetos_df["Título do Edital"].str.startswith("PIBIC JR", na=False),
    membros_projetos_df["Título do Edital"].str.startswith("PIBIC AF", na=False),
    membros_projetos_df["Título do Edital"].str.startswith("PIBIC", na=False),
    membros_projetos_df["Título do Edital"].str.contains("EXTERNO", na=False),
    membros_projetos_df["Título do Edital"].str.contains("INTERNO", na=False),
]

choices = [
    "PICTV",
    "PIBITI",
    "PIBIC JR AF",
    "PIBIC JR",
    "PIBIC AF",
    "PIBIC",
    "FOMENTO EXTERNO",
    "FOMENTO INTERNO"
]

membros_projetos_df["Edital"] = np.select(conditions, choices, default="OUTRO")
membros_projetos_df = membros_projetos_df[membros_projetos_df["Título do Edital"] != "."]

### Padronização da Categoria do Membro

In [ ]:
membros_projetos_df[["Categoria do Membro"]] = (
    membros_projetos_df[["Categoria do Membro"]]
        .fillna("Não Informado")
)

### Vinculo de Pesquisadores

In [ ]:
vinculo_pesquisadores_df = membros_projetos_df

vinculo_pesquisadores_df["Tem Bolsista"] = vinculo_pesquisadores_df["Vínculo com Projeto"].eq("Bolsista")
vinculo_pesquisadores_df["Tem Voluntario"] = vinculo_pesquisadores_df["Vínculo com Projeto"].eq("Voluntário")

vinculo_pesquisadores_df = (
    vinculo_pesquisadores_df.groupby(["ID do Membro", "Categoria do Membro"]).agg({
          "Tem Bolsista": "max",
          "Tem Voluntario": "max"
      })
      .reset_index()
)

vinculo_pesquisadores_df["Tipo de Participação"] = (
    vinculo_pesquisadores_df["Tem Bolsista"].map({True: "Bolsista", False: ""}) +
    vinculo_pesquisadores_df["Tem Voluntario"].map({True: "Voluntário", False: ""})
).str.replace("BolsistaVoluntário", "Bolsista e Voluntário")

## Projetos de Pesquisa

In [53]:
query = """
SELECT DISTINCT
    projeto.id AS cod_projeto,
    projeto.titulo AS titulo_projeto,
    uo.nome AS campus,
    projeto.aprovado, 
    projeto.inicio_execucao,
    EXTRACT(YEAR FROM projeto.inicio_execucao) AS ano_inicio_execucao,
    projeto.fim_execucao,
    EXTRACT(YEAR FROM projeto.fim_execucao) AS ano_fim_execucao,
    projeto.data_finalizacao_conclusao,
    gp.descricao as grupo_pesquisa_nome,
    ac.descricao as area_conhecimento,
    COALESCE(ac_superior.descricao, ac.descricao) AS grande_area_conhecimento,
    projeto.edital_id,
    edital.titulo AS titulo_edital,
    edital.tipo_edital as tipo_edital_id,


    CASE
        WHEN edital.tipo_edital = '1' THEN 'PESQUISA'
        WHEN edital.tipo_edital = '2' THEN 'INOVACAO'
        WHEN edital.tipo_edital = '3' THEN 'PESQUISA_INOVACAO_CONTINUO'
        WHEN edital.tipo_edital = '4' THEN 'EXTENSAO_FLUXO_CONTINUO'
        ELSE 'DESCONHECIDO'
    END AS descricao_tipo_edital,


    TRUE AS inscrito,


    CASE 
        WHEN projeto.data_conclusao_planejamento IS NULL
             AND edital.inicio_inscricoes <= NOW()
             AND edital.fim_inscricoes >= NOW()
        THEN TRUE ELSE FALSE
    END AS em_edicao,


    CASE 
        WHEN projeto.pre_aprovado = TRUE
        THEN TRUE ELSE FALSE
    END AS pre_aprovado,


    CASE 
        WHEN EXISTS (
            SELECT 1 
            FROM pesquisa_registroconclusaoprojeto r
            WHERE r.projeto_id = projeto.id
              AND r.dt_avaliacao IS NOT NULL
        )
        THEN TRUE ELSE FALSE
    END AS concluido,


    CASE 
        WHEN projeto.inativado = TRUE
        THEN TRUE ELSE FALSE
    END AS inativado,
	
    CASE 
        WHEN projeto.data_conclusao_planejamento IS NULL
             AND projeto.data_pre_avaliacao IS NULL
             AND edital.fim_inscricoes < NOW()
        THEN TRUE ELSE FALSE
    END AS nao_enviado,


    CASE 
        WHEN projeto.data_conclusao_planejamento IS NOT NULL
        THEN TRUE ELSE FALSE
    END AS enviado
FROM pesquisa_projeto projeto
JOIN pesquisa_edital edital 
    ON projeto.edital_id = edital.id
LEFT JOIN cnpq_grupopesquisa gp 
    ON projeto.grupo_pesquisa_id = gp.id
LEFT JOIN rh_areaconhecimento ac 
    ON projeto.area_conhecimento_id = ac.id
LEFT JOIN rh_areaconhecimento ac_superior 
    ON ac.superior_id = ac_superior.id
LEFT JOIN unidadeorganizacional uo 
    ON uo.id = projeto.uo_id


ORDER BY cod_projeto ASC;
"""

with engine.connect() as conn:
    projetos_df = pd.read_sql(text(query), conn)

projetos_df = projetos_df.rename(columns={
     "cod_projeto": "ID do Projeto",
    "titulo_projeto": "Título do Projeto",
    "campus": "Campus",
    "aprovado": "Aprovado",
    "inicio_execucao": "Início da Execução",
    "ano_inicio_execucao": "Ano de Início da Execução",
    "fim_execucao": "Fim da Execução",
    "ano_fim_execucao": "Ano de Fim da Execução",
    "grupo_pesquisa_nome": "Nome do Grupo de Pesquisa",
    "area_conhecimento": "Área de Conhecimento",
    "edital_id": "ID do Edital",
    "titulo_edital": "Título do Edital",
    "tipo_edital_id": "ID do Tipo de Edital",
    "descricao_tipo_edital": "Tipo de Edital",
    "grande_area_conhecimento": "Grande Área de Conhecimento",
    "data_finalizacao_conclusao": "Data de Finalização da Conclusão"
})

### Padronização de Campus do Projeto

In [54]:
projetos_df["Campus"] = projetos_df["Campus"].str.replace("CAMPUS ", "", regex=False)
projetos_df["Campus"] = projetos_df["Campus"].str.replace("AVANÇADO DE ", "", regex=False)

### Padronização de Título do Projeto

In [55]:
projetos_df['Título do Projeto'] = (
    projetos_df['Título do Projeto']
    .astype(str)
    .str.strip()
    .str.title()
    .str.replace(r'\s+', ' ', regex=True)
)

### Padronização de Edital

In [56]:
projetos_df["Título do Edital"] = projetos_df["Título do Edital"].str.upper()

conditions = [
    projetos_df["Título do Edital"].str.startswith("PROGRAMA DE INICIAÇÃO CIENTÍFICA E TECNOLÓGICA VOLUNTÁRIA", na=False),
    projetos_df["Título do Edital"].str.startswith("PIBITI", na=False),
    projetos_df["Título do Edital"].str.startswith("PIBIC JR AF", na=False),
    projetos_df["Título do Edital"].str.startswith("PIBIC JR", na=False),
    projetos_df["Título do Edital"].str.startswith("PIBIC AF", na=False),
    projetos_df["Título do Edital"].str.startswith("PIBIC", na=False),
    projetos_df["Título do Edital"].str.contains("EXTERNO", na=False),
    projetos_df["Título do Edital"].str.contains("INTERNO", na=False),
]

choices = [
    "PICTV",
    "PIBITI",
    "PIBIC JR AF",
    "PIBIC JR",
    "PIBIC AF",
    "PIBIC",
    "FOMENTO EXTERNO",
    "FOMENTO INTERNO"
]

projetos_df["Edital"] = np.select(conditions, choices, default="OUTRO")
projetos_df = projetos_df[projetos_df["Título do Edital"] != "."]

## Projetos Aprovados por Grupo de Pesquisa

In [77]:
query = """
SELECT
   grupo_pesquisa_id,
   grupo_pesquisa_nome,
   COUNT(*) AS total
FROM vw_painelprpi_projetos_pesquisa
WHERE aprovado = TRUE
GROUP BY grupo_pesquisa_id, grupo_pesquisa_nome
ORDER BY grupo_pesquisa_id;
"""

with engine.connect() as conn:
    grupos_pesquisa_df = pd.read_sql(text(query), conn)

grupos_pesquisa_df = grupos_pesquisa_df.rename(columns={
    "grupo_pesquisa_id": "ID do Grupo de Pesquisa",
    "grupo_pesquisa_nome": "Nome do Grupo de Pesquisa",
    "total": "Total de Projetos Aprovados"
})

grupos_pesquisa_df = grupos_pesquisa_df[grupos_pesquisa_df["Nome do Grupo de Pesquisa"].notna()]


## Orientações

In [83]:
query_concluidas = "SELECT * FROM public.cnpq_orientacaoconcluida"
query_em_andamento = "SELECT * FROM public.cnpq_orientacaoandamento"

with engine.connect() as conn:
    orientacoes_concluidas_df = pd.read_sql(text(query_concluidas), conn)
    orientacoes_andamento_df = pd.read_sql(text(query_em_andamento), conn)

orientacoes_andamento_df["Situação"] = "Em Andamento"
orientacoes_concluidas_df["Situação"] = "Concluida"

orientacoes_df = pd.concat(
    [orientacoes_concluidas_df, orientacoes_andamento_df],
    ignore_index=True
)

## Pesquisadores Vinculados a Programa de Pós-Graduação (PPG)

In [14]:
query = """
SELECT DISTINCT
  ppg.nome AS programa_pos_nome,
  pesq_partic.vinculo_pessoa_id AS pesquisador_vinculo_id,
  painelprpi_get_pessoa_nome(
      painelprpi_get_pessoa_id_from_vinculo(pesq_partic.vinculo_pessoa_id)
  ) AS pesquisador_nome,
  pesq_proj.id AS projeto_id,
  pesq_proj.titulo AS titulo_projeto,
  CASE
      WHEN pesq_partic.responsavel = TRUE THEN 'RESPONSÁVEL'
      ELSE 'PARTICIPANTE'
  END AS papel_no_projeto
FROM pesquisa_participacao pesq_partic
JOIN pesquisa_projeto pesq_proj
  ON pesq_proj.id = pesq_partic.projeto_id
JOIN pesquisa_programaposgraduacao ppg
  ON ppg.id = pesq_proj.ppg_id
WHERE pesq_proj.aprovado = TRUE
AND pesq_partic.vinculo_pessoa_id IS NOT NULL
ORDER BY   programa_pos_nome, pesquisador_nome, titulo_projeto;
"""

# Geral ✅

## Estatísticas

In [57]:
total_projetos = projetos_df.shape[0]
total_projetos_aprovados = projetos_df[projetos_df["Aprovado"] == True].shape[0]
total_areas_conhecimento = projetos_df["Área de Conhecimento"].nunique()
total_grupos_de_pesquisa = projetos_df["Nome do Grupo de Pesquisa"].nunique()
total_patentes = producoes_tecnicas_df[producoes_tecnicas_df["Tipo de Publicação"] == "Patente"].shape[0]
total_producoes_tecnicas = producoes_tecnicas_df.shape[0]

print(f"Total de Projetos: {total_projetos}")
print(f"Total de Projetos Aprovados: {total_projetos_aprovados}")
print(f"Total de Áreas de Conhecimento: {total_areas_conhecimento}")
print(f"Total de Grupos de Pesquisa: {total_grupos_de_pesquisa}")
print(f"Total de Patentes: {total_patentes}")
print(f"Total de Produções Técnicas: {total_producoes_tecnicas}")

Total de Projetos: 3460
Total de Projetos Aprovados: 1843
Total de Áreas de Conhecimento: 66
Total de Grupos de Pesquisa: 197
Total de Patentes: 554
Total de Produções Técnicas: 83658


## Pesquisadores

In [30]:
total_pesquisadores = vinculo_pesquisadores_df["ID do Membro"].count()
print(f"Total de Pesquisadores: {total_pesquisadores}")

analise_pesquisadores = vinculo_pesquisadores_df.groupby("Categoria do Membro").agg(
    total_pesquisadores=('ID do Membro', 'count'),
    bolsistas=('Tipo de Participação', lambda x: (x == 'Bolsista').sum()),
    voluntarios=('Tipo de Participação', lambda x: (x == 'Voluntário').sum()),
    bolsistas_voluntarios=('Tipo de Participação', lambda x: (x == 'Bolsista e Voluntário').sum())
).reset_index().rename(columns={'total_pesquisadores': 'Total de Pesquisadores', 'bolsistas': 'Bolsistas', 'voluntarios': 'Voluntários', 'bolsistas_voluntarios': 'Bolsistas e Voluntários'})

analise_pesquisadores = analise_pesquisadores.sort_values(by="Total de Pesquisadores", ascending=False)

analise_pesquisadores['Bolsistas'] = (
    analise_pesquisadores['Bolsistas'].astype(str) +
    ' (' + (analise_pesquisadores['Bolsistas'] / analise_pesquisadores['Total de Pesquisadores'] * 100).round(2).astype(str) + '%)'
)

analise_pesquisadores['Voluntários'] = (
    analise_pesquisadores['Voluntários'].astype(str) +
    ' (' + (analise_pesquisadores['Voluntários'] / analise_pesquisadores['Total de Pesquisadores'] * 100).round(2).astype(str) + '%)'
)

analise_pesquisadores['Bolsistas e Voluntários'] = (
    analise_pesquisadores['Bolsistas e Voluntários'].astype(str) +
    ' (' + (analise_pesquisadores['Bolsistas e Voluntários'] / analise_pesquisadores['Total de Pesquisadores'] * 100).round(2).astype(str) + '%)'
)

display(analise_pesquisadores)

Total de Pesquisadores: 3535


,Categoria do Membro,Total de Pesquisadores,Bolsistas,Voluntários,Bolsistas e Voluntários
0,Discente,2315,1176 (50.8%),970 (41.9%),169 (7.3%)
1,Docente,1033,53 (5.13%),797 (77.15%),183 (17.72%)
3,TAE,99,34 (34.34%),56 (56.57%),9 (9.09%)
2,Não Informado,88,29 (32.95%),56 (63.64%),3 (3.41%)


# Projetos Por Grande Área ✅

## Projetos por Grande Área

In [ ]:
analise_grande_area = projetos_df.groupby("Grande Área de Conhecimento").agg(
    total_projetos=("ID do Projeto", "count")
).reset_index().rename(columns={"total_projetos": "Total de Projetos"}).sort_values(by="Total de Projetos", ascending=False)

analise_grande_area['Total de Projetos'] = (
    analise_grande_area['Total de Projetos'].astype(str) + 
    ' (' + (analise_grande_area['Total de Projetos'] / analise_grande_area['Total de Projetos'].sum() * 100).round(2).astype(str) + '%)'
)

display(analise_grande_area)

,Grande Área de Conhecimento,Total de Projetos
3,CIÊNCIAS EXATAS E DA TERRA,863 (24.95%)
6,ENGENHARIAS,700 (20.24%)
0,CIÊNCIAS AGRÁRIAS,634 (18.33%)
8,MULTIDISCIPLINAR,311 (8.99%)
4,CIÊNCIAS HUMANAS,307 (8.88%)
7,"LINGUÍSTICA, LETRAS E ARTES",181 (5.23%)
2,CIÊNCIAS DA SAÚDE,174 (5.03%)
1,CIÊNCIAS BIOLÓGICAS,148 (4.28%)
5,CIÊNCIAS SOCIAIS APLICADAS,141 (4.08%)


## Grande Área por Ano

In [32]:
analise_grande_area_por_ano = projetos_df.groupby("Grande Área de Conhecimento").agg(
    total_2022=("Ano de Início da Execução", lambda x: (x == 2022).sum()),
    total_2023=("Ano de Início da Execução", lambda x: (x == 2023).sum()),
    total_2024=("Ano de Início da Execução", lambda x: (x == 2024).sum()),
    total_2025=("Ano de Início da Execução", lambda x: (x == 2025).sum()),
    total_2026=("Ano de Início da Execução", lambda x: (x == 2026).sum())
).reset_index().rename(columns={"total_2026": "2026", "total_2025": "2025", "total_2024": "2024", "total_2023": "2023", "total_2022": "2022"})

display(analise_grande_area_por_ano)

,Grande Área de Conhecimento,2022,2023,2024,2025,2026
0,CIÊNCIAS AGRÁRIAS,0,181,201,231,20
1,CIÊNCIAS BIOLÓGICAS,0,37,37,62,12
2,CIÊNCIAS DA SAÚDE,0,47,56,63,8
3,CIÊNCIAS EXATAS E DA TERRA,1,245,265,312,40
4,CIÊNCIAS HUMANAS,0,75,75,138,19
5,CIÊNCIAS SOCIAIS APLICADAS,2,38,32,65,4
6,ENGENHARIAS,1,214,218,245,22
7,"LINGUÍSTICA, LETRAS E ARTES",0,55,47,72,7
8,MULTIDISCIPLINAR,1,80,79,138,13


## Projetos por Campus

In [33]:
analise_projetos_por_campus = projetos_df.groupby("Campus").agg(
    total_projetos=("ID do Projeto", "count"),
).reset_index().rename(columns={"total_projetos": "Total de Projetos"})

analise_projetos_por_campus = analise_projetos_por_campus.sort_values(by="Total de Projetos", ascending=False)

analise_projetos_por_campus['Total de Projetos'] = (
    analise_projetos_por_campus['Total de Projetos'].astype(str) + 
    ' (' + (analise_projetos_por_campus['Total de Projetos'] / analise_projetos_por_campus['Total de Projetos'].sum() * 100).round(2).astype(str) + '%)'
)

display(analise_projetos_por_campus)

,Campus,Total de Projetos
11,FORTALEZA,366 (10.58%)
19,LIMOEIRO DO NORTE,333 (9.63%)
28,SOBRAL,217 (6.27%)
26,QUIXADA,196 (5.67%)
20,MARACANAÚ,195 (5.64%)
14,IGUATU,172 (4.97%)
31,TIANGUA,143 (4.13%)
6,CANINDE,138 (3.99%)
18,JUAZEIRO DO NORTE,136 (3.93%)
10,CRATO,120 (3.47%)


## Grande Área por Edital

In [36]:
analise_grande_area_por_edital = projetos_df.groupby("Grande Área de Conhecimento").agg(
    total_projetos=("ID do Projeto", "count"),
    total_externo=("Edital", lambda x: (x == "FOMENTO EXTERNO").sum()),
    total_interno=("Edital", lambda x: (x == "FOMENTO INTERNO").sum()),
    total_pibic=("Edital", lambda x: (x == "PIBIC").sum()),
    total_pibic_af=("Edital", lambda x: (x == "PIBIC AF").sum()),
    total_pibic_jr=("Edital", lambda x: (x == "PIBIC JR").sum()),
    total_pibic_jr_af=("Edital", lambda x: (x == "PIBIC JR AF").sum()),
    total_pictv=("Edital", lambda x: (x == "PICTV").sum()),
    total_pibiti=("Edital", lambda x: (x == "PIBITI").sum())
).reset_index().rename(columns={
    "total_projetos": "Total de Projetos",
    "total_externo": "Fomento Externo",
    "total_interno": "Fomento Interno",
    "total_pibic": "PIBIC",
    "total_pibic_af": "PIBIC AF",
    "total_pibic_jr": "PIBIC JR",
    "total_pibic_jr_af": "PIBIC JR AF",
    "total_pictv": "PICTV",
    "total_pibiti": "PIBITI"
})
analise_grande_area_por_edital = analise_grande_area_por_edital.sort_values(by="Total de Projetos", ascending=False)

analise_grande_area_por_edital['Fomento Externo'] = (
    analise_grande_area_por_edital['Fomento Externo'].astype(str) +
    ' (' + (analise_grande_area_por_edital['Fomento Externo'] / analise_grande_area_por_edital['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

analise_grande_area_por_edital['Fomento Interno'] = (
    analise_grande_area_por_edital['Fomento Interno'].astype(str) +
    ' (' + (analise_grande_area_por_edital['Fomento Interno'] / analise_grande_area_por_edital['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

analise_grande_area_por_edital['PIBIC'] = (
    analise_grande_area_por_edital['PIBIC'].astype(str) + 
    ' (' + (analise_grande_area_por_edital['PIBIC'] / analise_grande_area_por_edital['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

analise_grande_area_por_edital['PIBIC AF'] = (
    analise_grande_area_por_edital['PIBIC AF'].astype(str) +
    ' (' + (analise_grande_area_por_edital['PIBIC AF'] / analise_grande_area_por_edital['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

analise_grande_area_por_edital['PIBIC JR'] = (
    analise_grande_area_por_edital['PIBIC JR'].astype(str) +
    ' (' + (analise_grande_area_por_edital['PIBIC JR'] / analise_grande_area_por_edital['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

analise_grande_area_por_edital['PIBIC JR AF'] = (
    analise_grande_area_por_edital['PIBIC JR AF'].astype(str) +
    ' (' + (analise_grande_area_por_edital['PIBIC JR AF'] / analise_grande_area_por_edital['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

analise_grande_area_por_edital['PICTV'] = (
    analise_grande_area_por_edital['PICTV'].astype(str) +
    ' (' + (analise_grande_area_por_edital['PICTV'] / analise_grande_area_por_edital['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

analise_grande_area_por_edital['PIBITI'] = (
    analise_grande_area_por_edital['PIBITI'].astype(str) +
    ' (' + (analise_grande_area_por_edital['PIBITI'] / analise_grande_area_por_edital['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

display(analise_grande_area_por_edital)

,Grande Área de Conhecimento,Total de Projetos,Fomento Externo,Fomento Interno,PIBIC,PIBIC AF,PIBIC JR,PIBIC JR AF,PICTV,PIBITI
3,CIÊNCIAS EXATAS E DA TERRA,863,67 (7.76%),17 (1.97%),231 (26.77%),51 (5.91%),119 (13.79%),30 (3.48%),207 (23.99%),141 (16.34%)
6,ENGENHARIAS,700,28 (4.0%),12 (1.71%),208 (29.71%),46 (6.57%),123 (17.57%),22 (3.14%),124 (17.71%),137 (19.57%)
0,CIÊNCIAS AGRÁRIAS,634,10 (1.58%),5 (0.79%),217 (34.23%),73 (11.51%),88 (13.88%),27 (4.26%),78 (12.3%),136 (21.45%)
8,MULTIDISCIPLINAR,311,9 (2.89%),2 (0.64%),99 (31.83%),42 (13.5%),48 (15.43%),14 (4.5%),54 (17.36%),43 (13.83%)
4,CIÊNCIAS HUMANAS,307,4 (1.3%),8 (2.61%),98 (31.92%),35 (11.4%),35 (11.4%),20 (6.51%),86 (28.01%),21 (6.84%)
7,"LINGUÍSTICA, LETRAS E ARTES",181,2 (1.1%),5 (2.76%),77 (42.54%),12 (6.63%),16 (8.84%),4 (2.21%),59 (32.6%),6 (3.31%)
2,CIÊNCIAS DA SAÚDE,174,1 (0.57%),2 (1.15%),60 (34.48%),25 (14.37%),23 (13.22%),4 (2.3%),38 (21.84%),21 (12.07%)
1,CIÊNCIAS BIOLÓGICAS,148,4 (2.7%),1 (0.68%),51 (34.46%),19 (12.84%),13 (8.78%),3 (2.03%),43 (29.05%),14 (9.46%)
5,CIÊNCIAS SOCIAIS APLICADAS,141,4 (2.84%),5 (3.55%),46 (32.62%),14 (9.93%),22 (15.6%),9 (6.38%),30 (21.28%),11 (7.8%)


## Projetos Aprovados

### Por área de conhecimento

In [37]:
projetos_aprovados_df = projetos_df[projetos_df["Aprovado"] == True]

analise_grande_area_aprovados = projetos_aprovados_df.groupby("Grande Área de Conhecimento").agg(
    total_projetos=("ID do Projeto", "count")
).reset_index().rename(columns={"total_projetos": "Total de Projetos"}).sort_values(by="Total de Projetos", ascending=False)

analise_grande_area_aprovados['Total de Projetos'] = (
    analise_grande_area_aprovados['Total de Projetos'].astype(str) + 
    ' (' + (analise_grande_area_aprovados['Total de Projetos'] / analise_grande_area_aprovados['Total de Projetos'].sum() * 100).round(2).astype(str) + '%)'
)

display(analise_grande_area_aprovados)

,Grande Área de Conhecimento,Total de Projetos
3,CIÊNCIAS EXATAS E DA TERRA,460 (24.96%)
6,ENGENHARIAS,374 (20.29%)
0,CIÊNCIAS AGRÁRIAS,336 (18.23%)
8,MULTIDISCIPLINAR,183 (9.93%)
4,CIÊNCIAS HUMANAS,150 (8.14%)
7,"LINGUÍSTICA, LETRAS E ARTES",93 (5.05%)
1,CIÊNCIAS BIOLÓGICAS,92 (4.99%)
2,CIÊNCIAS DA SAÚDE,78 (4.23%)
5,CIÊNCIAS SOCIAIS APLICADAS,77 (4.18%)


### Por ano

In [39]:
analise_grande_area_por_ano_aprovado = projetos_aprovados_df.groupby("Grande Área de Conhecimento").agg(
    total_2022=("Ano de Início da Execução", lambda x: (x == 2022).sum()),
    total_2023=("Ano de Início da Execução", lambda x: (x == 2023).sum()),
    total_2024=("Ano de Início da Execução", lambda x: (x == 2024).sum()),
    total_2025=("Ano de Início da Execução", lambda x: (x == 2025).sum()),
    total_2026=("Ano de Início da Execução", lambda x: (x == 2026).sum())
).reset_index().rename(columns={"total_2026": "2026", "total_2025": "2025", "total_2024": "2024", "total_2023": "2023", "total_2022": "2022"})

display(analise_grande_area_por_ano_aprovado)

,Grande Área de Conhecimento,2022,2023,2024,2025,2026
0,CIÊNCIAS AGRÁRIAS,0,112,112,105,7
1,CIÊNCIAS BIOLÓGICAS,0,26,18,42,6
2,CIÊNCIAS DA SAÚDE,0,33,23,21,1
3,CIÊNCIAS EXATAS E DA TERRA,0,143,132,168,17
4,CIÊNCIAS HUMANAS,0,41,35,64,10
5,CIÊNCIAS SOCIAIS APLICADAS,1,24,20,30,2
6,ENGENHARIAS,0,120,114,130,10
7,"LINGUÍSTICA, LETRAS E ARTES",0,39,22,28,4
8,MULTIDISCIPLINAR,1,56,50,71,5


### Por campus

In [40]:
analise_projetos_aprovados_por_campus = projetos_aprovados_df.groupby("Campus").agg(
    total_projetos=("ID do Projeto", "count"),
).reset_index().rename(columns={"total_projetos": "Total de Projetos"})

analise_projetos_aprovados_por_campus = analise_projetos_aprovados_por_campus.sort_values(by="Total de Projetos", ascending=False)

analise_projetos_aprovados_por_campus['Total de Projetos'] = (
    analise_projetos_aprovados_por_campus['Total de Projetos'].astype(str) + 
    ' (' + (analise_projetos_aprovados_por_campus['Total de Projetos'] / analise_projetos_aprovados_por_campus['Total de Projetos'].sum() * 100).round(2).astype(str) + '%)'
)

display(analise_projetos_aprovados_por_campus)

,Campus,Total de Projetos
11,FORTALEZA,231 (12.53%)
19,LIMOEIRO DO NORTE,201 (10.91%)
28,SOBRAL,132 (7.16%)
20,MARACANAÚ,106 (5.75%)
26,QUIXADA,103 (5.59%)
14,IGUATU,85 (4.61%)
31,TIANGUA,81 (4.4%)
6,CANINDE,65 (3.53%)
18,JUAZEIRO DO NORTE,65 (3.53%)
24,PARACURU,54 (2.93%)


### Por Edital

In [41]:
analise_grande_area_por_edital_aprovados = projetos_aprovados_df.groupby("Grande Área de Conhecimento").agg(
    total_projetos=("ID do Projeto", "count"),
    total_externo=("Edital", lambda x: (x == "FOMENTO EXTERNO").sum()),
    total_interno=("Edital", lambda x: (x == "FOMENTO INTERNO").sum()),
    total_pibic=("Edital", lambda x: (x == "PIBIC").sum()),
    total_pibic_af=("Edital", lambda x: (x == "PIBIC AF").sum()),
    total_pibic_jr=("Edital", lambda x: (x == "PIBIC JR").sum()),
    total_pibic_jr_af=("Edital", lambda x: (x == "PIBIC JR AF").sum()),
    total_pictv=("Edital", lambda x: (x == "PICTV").sum()),
    total_pibiti=("Edital", lambda x: (x == "PIBITI").sum())
).reset_index().rename(columns={
    "total_projetos": "Total de Projetos",
    "total_externo": "Fomento Externo",
    "total_interno": "Fomento Interno",
    "total_pibic": "PIBIC",
    "total_pibic_af": "PIBIC AF",
    "total_pibic_jr": "PIBIC JR",
    "total_pibic_jr_af": "PIBIC JR AF",
    "total_pictv": "PICTV",
    "total_pibiti": "PIBITI"
})
analise_grande_area_por_edital_aprovados = analise_grande_area_por_edital_aprovados.sort_values(by="Total de Projetos", ascending=False)

analise_grande_area_por_edital_aprovados['Fomento Externo'] = (
    analise_grande_area_por_edital_aprovados['Fomento Externo'].astype(str) +
    ' (' + (analise_grande_area_por_edital_aprovados['Fomento Externo'] / analise_grande_area_por_edital_aprovados['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

analise_grande_area_por_edital_aprovados['Fomento Interno'] = (
    analise_grande_area_por_edital_aprovados['Fomento Interno'].astype(str) +
    ' (' + (analise_grande_area_por_edital_aprovados['Fomento Interno'] / analise_grande_area_por_edital_aprovados['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

analise_grande_area_por_edital_aprovados['PIBIC'] = (
    analise_grande_area_por_edital_aprovados['PIBIC'].astype(str) + 
    ' (' + (analise_grande_area_por_edital_aprovados['PIBIC'] / analise_grande_area_por_edital_aprovados['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

analise_grande_area_por_edital_aprovados['PIBIC AF'] = (
    analise_grande_area_por_edital_aprovados['PIBIC AF'].astype(str) +
    ' (' + (analise_grande_area_por_edital_aprovados['PIBIC AF'] / analise_grande_area_por_edital_aprovados['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

analise_grande_area_por_edital_aprovados['PIBIC JR'] = (
    analise_grande_area_por_edital_aprovados['PIBIC JR'].astype(str) +
    ' (' + (analise_grande_area_por_edital_aprovados['PIBIC JR'] / analise_grande_area_por_edital_aprovados['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

analise_grande_area_por_edital_aprovados['PIBIC JR AF'] = (
    analise_grande_area_por_edital_aprovados['PIBIC JR AF'].astype(str) +
    ' (' + (analise_grande_area_por_edital_aprovados['PIBIC JR AF'] / analise_grande_area_por_edital_aprovados['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

analise_grande_area_por_edital_aprovados['PICTV'] = (
    analise_grande_area_por_edital_aprovados['PICTV'].astype(str) +
    ' (' + (analise_grande_area_por_edital_aprovados['PICTV'] / analise_grande_area_por_edital_aprovados['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

analise_grande_area_por_edital_aprovados['PIBITI'] = (
    analise_grande_area_por_edital_aprovados['PIBITI'].astype(str) +
    ' (' + (analise_grande_area_por_edital_aprovados['PIBITI'] / analise_grande_area_por_edital_aprovados['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

display(analise_grande_area_por_edital_aprovados)

,Grande Área de Conhecimento,Total de Projetos,Fomento Externo,Fomento Interno,PIBIC,PIBIC AF,PIBIC JR,PIBIC JR AF,PICTV,PIBITI
3,CIÊNCIAS EXATAS E DA TERRA,460,39 (8.48%),13 (2.83%),106 (23.04%),10 (2.17%),71 (15.43%),16 (3.48%),152 (33.04%),53 (11.52%)
6,ENGENHARIAS,374,21 (5.61%),8 (2.14%),98 (26.2%),9 (2.41%),74 (19.79%),14 (3.74%),95 (25.4%),55 (14.71%)
0,CIÊNCIAS AGRÁRIAS,336,6 (1.79%),4 (1.19%),107 (31.85%),24 (7.14%),54 (16.07%),18 (5.36%),49 (14.58%),74 (22.02%)
8,MULTIDISCIPLINAR,183,5 (2.73%),1 (0.55%),61 (33.33%),17 (9.29%),27 (14.75%),7 (3.83%),35 (19.13%),30 (16.39%)
4,CIÊNCIAS HUMANAS,150,3 (2.0%),4 (2.67%),43 (28.67%),6 (4.0%),17 (11.33%),6 (4.0%),57 (38.0%),14 (9.33%)
7,"LINGUÍSTICA, LETRAS E ARTES",93,0 (0.0%),2 (2.15%),30 (32.26%),1 (1.08%),10 (10.75%),4 (4.3%),43 (46.24%),3 (3.23%)
1,CIÊNCIAS BIOLÓGICAS,92,4 (4.35%),1 (1.09%),27 (29.35%),7 (7.61%),9 (9.78%),2 (2.17%),33 (35.87%),9 (9.78%)
2,CIÊNCIAS DA SAÚDE,78,1 (1.28%),1 (1.28%),25 (32.05%),6 (7.69%),14 (17.95%),2 (2.56%),22 (28.21%),7 (8.97%)
5,CIÊNCIAS SOCIAIS APLICADAS,77,2 (2.6%),2 (2.6%),20 (25.97%),4 (5.19%),14 (18.18%),4 (5.19%),25 (32.47%),6 (7.79%)


## Status de Projetos

### Estatísticas

In [58]:
projetos_concluidos = projetos_df[projetos_df["concluido"] == True].shape[0]
projetos_inscritos = projetos_df[projetos_df["inscrito"] == True].shape[0]
projetos_em_edicao = projetos_df[projetos_df["em_edicao"] == True].shape[0]
projetos_pre_aprovados = projetos_df[projetos_df["pre_aprovado"] == True].shape[0]
projetos_inativados = projetos_df[projetos_df["inativado"] == True].shape[0]
projetos_nao_enviados = projetos_df[projetos_df["nao_enviado"] == True].shape[0]
projetos_enviados = projetos_df[projetos_df["enviado"] == True].shape[0]
projetos_aprovados = projetos_df[projetos_df["Aprovado"] == True].shape[0]

print(f"Total de Projetos Concluídos: {projetos_concluidos}" + f" ({projetos_concluidos / total_projetos * 100:.2f}%)")
print(f"Total de Projetos Inscritos: {projetos_inscritos}" + f" ({projetos_inscritos / total_projetos * 100:.2f}%)")
print(f"Total de Projetos em Edição: {projetos_em_edicao}" + f" ({projetos_em_edicao / total_projetos * 100:.2f}%)")
print(f"Total de Projetos Pré-Aprovados: {projetos_pre_aprovados}" + f" ({projetos_pre_aprovados / total_projetos * 100:.2f}%)")
print(f"Total de Projetos Inativados: {projetos_inativados}" + f" ({projetos_inativados / total_projetos * 100:.2f}%)")
print(f"Total de Projetos Não Enviados: {projetos_nao_enviados}" + f" ({projetos_nao_enviados / total_projetos * 100:.2f}%)")
print(f"Total de Projetos Enviados: {projetos_enviados}" + f" ({projetos_enviados / total_projetos * 100:.2f}%)")
print(f"Total de Projetos Aprovados: {projetos_aprovados}" + f" ({projetos_aprovados / total_projetos * 100:.2f}%)")

Total de Projetos Concluídos: 1002 (28.96%)
Total de Projetos Inscritos: 3460 (100.00%)
Total de Projetos em Edição: 44 (1.27%)
Total de Projetos Pré-Aprovados: 2860 (82.66%)
Total de Projetos Inativados: 172 (4.97%)
Total de Projetos Não Enviados: 421 (12.17%)
Total de Projetos Enviados: 2987 (86.33%)
Total de Projetos Aprovados: 1843 (53.27%)


### Por Grande Área

In [70]:
analise_grande_area_status = projetos_df.groupby("Grande Área de Conhecimento").agg(
    projetos_inscritos=("ID do Projeto", "count"),
    projetos_concluidos=("concluido", "sum"),
    projetos_em_edicao=("em_edicao", "sum"),
    projetos_pre_aprovados=("pre_aprovado", "sum"),
    projetos_inativados=("inativado", "sum"),
    projetos_nao_enviados=("nao_enviado", "sum"),
    projetos_enviados=("enviado", "sum"),
    projetos_aprovados=("Aprovado", "sum")
).reset_index().rename(columns={
    "projetos_inscritos": "Projetos Inscritos",
    "projetos_concluidos": "Projetos Concluídos",
    "projetos_em_edicao": "Projetos em Edição",
    "projetos_pre_aprovados": "Projetos Pré-Aprovados",
    "projetos_inativados": "Projetos Inativados",
    "projetos_nao_enviados": "Projetos Não Enviados",
    "projetos_enviados": "Projetos Enviados",
    "projetos_aprovados": "Projetos Aprovados"
})

analise_grande_area_status['Projetos Concluídos'] = (
    analise_grande_area_status['Projetos Concluídos'].astype(str) + 
    ' (' + (analise_grande_area_status['Projetos Concluídos'] / analise_grande_area_status['Projetos Inscritos'] * 100).round(2).astype(str) + '%)'
)

analise_grande_area_status['Projetos em Edição'] = (
    analise_grande_area_status['Projetos em Edição'].astype(str) + 
    ' (' + (analise_grande_area_status['Projetos em Edição'] / analise_grande_area_status['Projetos Inscritos'] * 100).round(2).astype(str) + '%)'
)

analise_grande_area_status['Projetos Pré-Aprovados'] = (
    analise_grande_area_status['Projetos Pré-Aprovados'].astype(str) +
    ' (' + (analise_grande_area_status['Projetos Pré-Aprovados'] / analise_grande_area_status['Projetos Inscritos'] * 100).round(2).astype(str) + '%)'
)

analise_grande_area_status['Projetos Inativados'] = (
    analise_grande_area_status['Projetos Inativados'].astype(str) +
    ' (' + (analise_grande_area_status['Projetos Inativados'] / analise_grande_area_status['Projetos Inscritos'] * 100).round(2).astype(str) + '%)'
)

analise_grande_area_status['Projetos Não Enviados'] = (
    analise_grande_area_status['Projetos Não Enviados'].astype(str) +
    ' (' + (analise_grande_area_status['Projetos Não Enviados'] / analise_grande_area_status['Projetos Inscritos'] * 100).round(2).astype(str) + '%)'
)

analise_grande_area_status['Projetos Enviados'] = (
    analise_grande_area_status['Projetos Enviados'].astype(str) +
    ' (' + (analise_grande_area_status['Projetos Enviados'] / analise_grande_area_status['Projetos Inscritos'] * 100).round(2).astype(str) + '%)' 
)

analise_grande_area_status['Projetos Aprovados'] = (
    analise_grande_area_status['Projetos Aprovados'].astype(str) +
    ' (' + (analise_grande_area_status['Projetos Aprovados'] / analise_grande_area_status['Projetos Inscritos'] * 100).round(2).astype(str) + '%)'
)

display(analise_grande_area_status)

,Grande Área de Conhecimento,Projetos Inscritos,Projetos Concluídos,Projetos em Edição,Projetos Pré-Aprovados,Projetos Inativados,Projetos Não Enviados,Projetos Enviados,Projetos Aprovados
0,CIÊNCIAS AGRÁRIAS,634,203 (32.02%),10 (1.58%),542 (85.49%),33 (5.21%),67 (10.57%),555 (87.54%),336 (53.0%)
1,CIÊNCIAS BIOLÓGICAS,148,43 (29.05%),1 (0.68%),126 (85.14%),7 (4.73%),10 (6.76%),137 (92.57%),92 (62.16%)
2,CIÊNCIAS DA SAÚDE,174,52 (29.89%),4 (2.3%),138 (79.31%),5 (2.87%),23 (13.22%),147 (84.48%),78 (44.83%)
3,CIÊNCIAS EXATAS E DA TERRA,863,235 (27.23%),10 (1.16%),709 (82.16%),41 (4.75%),111 (12.86%),741 (85.86%),460 (53.3%)
4,CIÊNCIAS HUMANAS,307,64 (20.85%),9 (2.93%),240 (78.18%),18 (5.86%),49 (15.96%),248 (80.78%),150 (48.86%)
5,CIÊNCIAS SOCIAIS APLICADAS,142,37 (26.06%),1 (0.7%),113 (79.58%),7 (4.93%),24 (16.9%),117 (82.39%),77 (54.23%)
6,ENGENHARIAS,700,212 (30.29%),6 (0.86%),590 (84.29%),37 (5.29%),68 (9.71%),623 (89.0%),374 (53.43%)
7,"LINGUÍSTICA, LETRAS E ARTES",181,56 (30.94%),1 (0.55%),150 (82.87%),14 (7.73%),24 (13.26%),155 (85.64%),93 (51.38%)
8,MULTIDISCIPLINAR,311,100 (32.15%),2 (0.64%),252 (81.03%),10 (3.22%),45 (14.47%),264 (84.89%),183 (58.84%)


# Projetos por Área de Conhecimento ✅

## Total de Projetos por Área de Conhecimento

In [62]:
analise_area_conhecimento = projetos_df.groupby("Área de Conhecimento").agg(
    total_projetos=("Título do Projeto", "nunique"),
).reset_index().rename(columns={"total_projetos": "Total de Projetos"}).sort_values(by="Total de Projetos", ascending=False)

display(analise_area_conhecimento)

,Área de Conhecimento,Total de Projetos
11,CIÊNCIA DA COMPUTAÇÃO,432
30,ENGENHARIA ELÉTRICA,232
1,AGRONOMIA,208
57,QUÍMICA,173
34,ENGENHARIA SANITÁRIA,170
13,CIÊNCIA E TECNOLOGIA DE ALIMENTOS,161
20,EDUCAÇÃO,153
44,INTERDISCIPLINAR,137
21,EDUCAÇÃO FÍSICA,111
35,ENSINO,94


## Área de Conhecimento Predominante por Campus

In [63]:
analise_area_conhecimento_por_campus = (
    projetos_df
    .groupby(["Campus", "Área de Conhecimento"])
    .agg(total_projetos=("Título do Projeto", "count"))
    .reset_index()
    .rename(columns={"total_projetos": "Total de Projetos"})
    .sort_values(["Campus", "Total de Projetos"], ascending=[True, False])
)

analise_area_conhecimento_por_campus = (
    analise_area_conhecimento_por_campus
    .loc[analise_area_conhecimento_por_campus.groupby("Campus")["Total de Projetos"].idxmax()]
    .reset_index(drop=True)
)

display(analise_area_conhecimento_por_campus)

,Campus,Área de Conhecimento,Total de Projetos
0,ACOPIARA,CIÊNCIA DA COMPUTAÇÃO,10
1,ACARAU,RECURSOS PESQUEIROS E ENGENHARIA DE PESCA,12
2,ARACATI,CIÊNCIA DA COMPUTAÇÃO,47
3,BATURITE,CIÊNCIA E TECNOLOGIA DE ALIMENTOS,11
4,BOA VIAGEM,ZOOTECNIA,33
5,CAMOCIM,MICROBIOLOGIA,15
6,CANINDE,EDUCAÇÃO FÍSICA,39
7,CAUCAIA,MATEMÁTICA,11
8,CEDRO,ENGENHARIA ELÉTRICA,33
9,CRATEUS,QUÍMICA,15


## Tipo de Edital por Área de Conhecimento

In [64]:
analise_edital_por_area_conhecimento = projetos_df.groupby("Área de Conhecimento").agg(
    total_projetos=("Título do Projeto", "count"),
    total_externo=("Edital", lambda x: (x == "FOMENTO EXTERNO").sum()),
    total_interno=("Edital", lambda x: (x == "FOMENTO INTERNO").sum()),
    total_pibic=("Edital", lambda x: (x == "PIBIC").sum()),
    total_pibic_af=("Edital", lambda x: (x == "PIBIC AF").sum()),
    total_pibic_jr=("Edital", lambda x: (x == "PIBIC JR").sum()),
    total_pibic_jr_af=("Edital", lambda x: (x == "PIBIC JR AF").sum()),
    total_pictv=("Edital", lambda x: (x == "PICTV").sum()),
    total_pibiti=("Edital", lambda x: (x == "PIBITI").sum())
).reset_index().rename(columns={
    "total_projetos": "Total de Projetos",
    "total_externo": "Fomento Externo",
    "total_interno": "Fomento Interno",
    "total_pibic": "PIBIC",
    "total_pibic_af": "PIBIC AF",
    "total_pibic_jr": "PIBIC JR",
    "total_pibic_jr_af": "PIBIC JR AF",
    "total_pictv": "PICTV",
    "total_pibiti": "PIBITI"
}).sort_values(by="Total de Projetos", ascending=False)

analise_edital_por_area_conhecimento['Fomento Externo'] = (
    analise_edital_por_area_conhecimento['Fomento Externo'].astype(str) +
    " (" + (analise_edital_por_area_conhecimento['Fomento Externo'] / analise_edital_por_area_conhecimento['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

analise_edital_por_area_conhecimento['Fomento Interno'] = (
    analise_edital_por_area_conhecimento['Fomento Interno'].astype(str) +
    " (" + (analise_edital_por_area_conhecimento['Fomento Interno'] / analise_edital_por_area_conhecimento['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

analise_edital_por_area_conhecimento['PIBIC'] = (
    analise_edital_por_area_conhecimento['PIBIC'].astype(str) +
      " (" + (analise_edital_por_area_conhecimento['PIBIC'] / analise_edital_por_area_conhecimento['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)  

analise_edital_por_area_conhecimento['PIBIC AF'] = (
    analise_edital_por_area_conhecimento['PIBIC AF'].astype(str) + 
    " (" + (analise_edital_por_area_conhecimento['PIBIC AF'] / analise_edital_por_area_conhecimento['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

analise_edital_por_area_conhecimento['PIBIC JR'] = (
    analise_edital_por_area_conhecimento['PIBIC JR'].astype(str) + 
    " (" + (analise_edital_por_area_conhecimento['PIBIC JR'] / analise_edital_por_area_conhecimento['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

analise_edital_por_area_conhecimento['PIBIC JR AF'] = (
    analise_edital_por_area_conhecimento['PIBIC JR AF'].astype(str) +
      " (" + (analise_edital_por_area_conhecimento['PIBIC JR AF'] / analise_edital_por_area_conhecimento['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

analise_edital_por_area_conhecimento['PICTV'] = (
    analise_edital_por_area_conhecimento['PICTV'].astype(str) + " (" +
      (analise_edital_por_area_conhecimento['PICTV'] / analise_edital_por_area_conhecimento['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

analise_edital_por_area_conhecimento['PIBITI'] = (
    analise_edital_por_area_conhecimento['PIBITI'].astype(str) + " (" +
      (analise_edital_por_area_conhecimento['PIBITI'] / analise_edital_por_area_conhecimento['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

display(analise_edital_por_area_conhecimento)

,Área de Conhecimento,Total de Projetos,Fomento Externo,Fomento Interno,PIBIC,PIBIC AF,PIBIC JR,PIBIC JR AF,PICTV,PIBITI
11,CIÊNCIA DA COMPUTAÇÃO,496,59 (11.9%),15 (3.02%),111 (22.38%),33 (6.65%),56 (11.29%),12 (2.42%),116 (23.39%),94 (18.95%)
30,ENGENHARIA ELÉTRICA,295,17 (5.76%),7 (2.37%),88 (29.83%),15 (5.08%),56 (18.98%),11 (3.73%),28 (9.49%),73 (24.75%)
1,AGRONOMIA,228,1 (0.44%),4 (1.75%),78 (34.21%),35 (15.35%),35 (15.35%),7 (3.07%),26 (11.4%),42 (18.42%)
57,QUÍMICA,192,3 (1.56%),1 (0.52%),56 (29.17%),9 (4.69%),37 (19.27%),14 (7.29%),43 (22.4%),29 (15.1%)
34,ENGENHARIA SANITÁRIA,191,1 (0.52%),1 (0.52%),65 (34.03%),15 (7.85%),32 (16.75%),6 (3.14%),33 (17.28%),38 (19.9%)
20,EDUCAÇÃO,191,3 (1.57%),6 (3.14%),65 (34.03%),21 (10.99%),16 (8.38%),8 (4.19%),60 (31.41%),12 (6.28%)
13,CIÊNCIA E TECNOLOGIA DE ALIMENTOS,190,0 (0.0%),0 (0.0%),65 (34.21%),20 (10.53%),21 (11.05%),9 (4.74%),17 (8.95%),58 (30.53%)
44,INTERDISCIPLINAR,155,3 (1.94%),2 (1.29%),47 (30.32%),24 (15.48%),24 (15.48%),8 (5.16%),15 (9.68%),32 (20.65%)
21,EDUCAÇÃO FÍSICA,127,1 (0.79%),2 (1.57%),47 (37.01%),20 (15.75%),18 (14.17%),2 (1.57%),20 (15.75%),17 (13.39%)
65,ZOOTECNIA,106,6 (5.66%),0 (0.0%),34 (32.08%),10 (9.43%),19 (17.92%),8 (7.55%),17 (16.04%),12 (11.32%)


## Projetos Aprovados

### Por Área de Conhecimento

In [65]:
analise_area_conhecimento_aprovados = projetos_aprovados_df.groupby("Área de Conhecimento").agg(
    total_projetos=("ID do Projeto", "count")
).reset_index().rename(columns={"total_projetos": "Total de Projetos"}).sort_values(by="Total de Projetos", ascending=False)

analise_area_conhecimento_aprovados['Total de Projetos'] = (
    analise_area_conhecimento_aprovados['Total de Projetos'].astype(str) + 
    ' (' + (analise_area_conhecimento_aprovados['Total de Projetos'] / analise_area_conhecimento_aprovados['Total de Projetos'].sum() * 100).round(2).astype(str) + '%)'
)

display(analise_area_conhecimento_aprovados)

,Área de Conhecimento,Total de Projetos
10,CIÊNCIA DA COMPUTAÇÃO,274 (14.87%)
27,ENGENHARIA ELÉTRICA,137 (7.43%)
1,AGRONOMIA,119 (6.46%)
31,ENGENHARIA SANITÁRIA,117 (6.35%)
12,CIÊNCIA E TECNOLOGIA DE ALIMENTOS,116 (6.29%)
53,QUÍMICA,114 (6.19%)
41,INTERDISCIPLINAR,99 (5.37%)
18,EDUCAÇÃO,87 (4.72%)
23,ENGENHARIA CIVIL,71 (3.85%)
32,ENSINO,62 (3.36%)


### Por Ano

In [66]:
analise_area_conhecimento_por_ano_aprovado = projetos_aprovados_df.groupby("Área de Conhecimento").agg(
    total_2022=("Ano de Início da Execução", lambda x: (x == 2022).sum()),
    total_2023=("Ano de Início da Execução", lambda x: (x == 2023).sum()),
    total_2024=("Ano de Início da Execução", lambda x: (x == 2024).sum()),
    total_2025=("Ano de Início da Execução", lambda x: (x == 2025).sum()),
    total_2026=("Ano de Início da Execução", lambda x: (x == 2026).sum())
).reset_index().rename(columns={"total_2026": "2026", "total_2025": "2025", "total_2024": "2024", "total_2023": "2023", "total_2022": "2022"})

display(analise_area_conhecimento_por_ano_aprovado)

,Área de Conhecimento,2022,2023,2024,2025,2026
0,ADMINISTRAÇÃO,0,7,9,11,1
1,AGRONOMIA,0,42,41,33,3
2,ANTROPOLOGIA,0,1,1,1,0
3,ARQUITETURA E URBANISMO,1,6,5,2,0
4,ARTES,0,10,5,5,2
5,ASTRONOMIA,0,2,2,4,0
6,BIOLOGIA GERAL,0,7,2,8,0
7,BIOQUÍMICA,0,1,1,2,0
8,BIOTECNOLOGIA,0,6,7,4,0
9,BOTÂNICA,0,4,3,5,1


### Por Edital

In [67]:
analise_area_conhecimento_por_edital_aprovados = projetos_aprovados_df.groupby("Área de Conhecimento").agg(
    total_projetos=("ID do Projeto", "count"),
    total_externo=("Edital", lambda x: (x == "FOMENTO EXTERNO").sum()),
    total_interno=("Edital", lambda x: (x == "FOMENTO INTERNO").sum()),
    total_pibic=("Edital", lambda x: (x == "PIBIC").sum()),
    total_pibic_af=("Edital", lambda x: (x == "PIBIC AF").sum()),
    total_pibic_jr=("Edital", lambda x: (x == "PIBIC JR").sum()),
    total_pibic_jr_af=("Edital", lambda x: (x == "PIBIC JR AF").sum()),
    total_pictv=("Edital", lambda x: (x == "PICTV").sum()),
    total_pibiti=("Edital", lambda x: (x == "PIBITI").sum())
).reset_index().rename(columns={
    "total_projetos": "Total de Projetos",
    "total_externo": "Fomento Externo",
    "total_interno": "Fomento Interno",
    "total_pibic": "PIBIC",
    "total_pibic_af": "PIBIC AF",
    "total_pibic_jr": "PIBIC JR",
    "total_pibic_jr_af": "PIBIC JR AF",
    "total_pictv": "PICTV",
    "total_pibiti": "PIBITI"
})
analise_area_conhecimento_por_edital_aprovados = analise_area_conhecimento_por_edital_aprovados.sort_values(by="Total de Projetos", ascending=False)

analise_area_conhecimento_por_edital_aprovados['Fomento Externo'] = (
    analise_area_conhecimento_por_edital_aprovados['Fomento Externo'].astype(str) +
    ' (' + (analise_area_conhecimento_por_edital_aprovados['Fomento Externo'] / analise_area_conhecimento_por_edital_aprovados['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

analise_area_conhecimento_por_edital_aprovados['Fomento Interno'] = (
    analise_area_conhecimento_por_edital_aprovados['Fomento Interno'].astype(str) +
    ' (' + (analise_area_conhecimento_por_edital_aprovados['Fomento Interno'] / analise_area_conhecimento_por_edital_aprovados['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

analise_area_conhecimento_por_edital_aprovados['PIBIC'] = (
    analise_area_conhecimento_por_edital_aprovados['PIBIC'].astype(str) + 
    ' (' + (analise_area_conhecimento_por_edital_aprovados['PIBIC'] / analise_area_conhecimento_por_edital_aprovados['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

analise_area_conhecimento_por_edital_aprovados['PIBIC AF'] = (
    analise_area_conhecimento_por_edital_aprovados['PIBIC AF'].astype(str) +
    ' (' + (analise_area_conhecimento_por_edital_aprovados['PIBIC AF'] / analise_area_conhecimento_por_edital_aprovados['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

analise_area_conhecimento_por_edital_aprovados['PIBIC JR'] = (
    analise_area_conhecimento_por_edital_aprovados['PIBIC JR'].astype(str) +
    ' (' + (analise_area_conhecimento_por_edital_aprovados['PIBIC JR'] / analise_area_conhecimento_por_edital_aprovados['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

analise_area_conhecimento_por_edital_aprovados['PIBIC JR AF'] = (
    analise_area_conhecimento_por_edital_aprovados['PIBIC JR AF'].astype(str) +
    ' (' + (analise_area_conhecimento_por_edital_aprovados['PIBIC JR AF'] / analise_area_conhecimento_por_edital_aprovados['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

analise_area_conhecimento_por_edital_aprovados['PICTV'] = (
    analise_area_conhecimento_por_edital_aprovados['PICTV'].astype(str) +
    ' (' + (analise_area_conhecimento_por_edital_aprovados['PICTV'] / analise_area_conhecimento_por_edital_aprovados['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

analise_area_conhecimento_por_edital_aprovados['PIBITI'] = (
    analise_area_conhecimento_por_edital_aprovados['PIBITI'].astype(str) +
    ' (' + (analise_area_conhecimento_por_edital_aprovados['PIBITI'] / analise_area_conhecimento_por_edital_aprovados['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

display(analise_area_conhecimento_por_edital_aprovados)

,Área de Conhecimento,Total de Projetos,Fomento Externo,Fomento Interno,PIBIC,PIBIC AF,PIBIC JR,PIBIC JR AF,PICTV,PIBITI
10,CIÊNCIA DA COMPUTAÇÃO,274,33 (12.04%),12 (4.38%),54 (19.71%),8 (2.92%),34 (12.41%),6 (2.19%),87 (31.75%),40 (14.6%)
27,ENGENHARIA ELÉTRICA,137,13 (9.49%),4 (2.92%),38 (27.74%),0 (0.0%),37 (27.01%),5 (3.65%),19 (13.87%),21 (15.33%)
1,AGRONOMIA,119,0 (0.0%),3 (2.52%),35 (29.41%),11 (9.24%),23 (19.33%),6 (5.04%),17 (14.29%),24 (20.17%)
31,ENGENHARIA SANITÁRIA,117,1 (0.85%),0 (0.0%),34 (29.06%),5 (4.27%),22 (18.8%),6 (5.13%),24 (20.51%),25 (21.37%)
12,CIÊNCIA E TECNOLOGIA DE ALIMENTOS,116,0 (0.0%),0 (0.0%),43 (37.07%),7 (6.03%),13 (11.21%),8 (6.9%),12 (10.34%),33 (28.45%)
53,QUÍMICA,114,3 (2.63%),1 (0.88%),33 (28.95%),1 (0.88%),25 (21.93%),9 (7.89%),30 (26.32%),12 (10.53%)
41,INTERDISCIPLINAR,99,2 (2.02%),1 (1.01%),35 (35.35%),8 (8.08%),17 (17.17%),5 (5.05%),9 (9.09%),22 (22.22%)
18,EDUCAÇÃO,87,3 (3.45%),2 (2.3%),24 (27.59%),5 (5.75%),8 (9.2%),1 (1.15%),35 (40.23%),9 (10.34%)
23,ENGENHARIA CIVIL,71,2 (2.82%),1 (1.41%),15 (21.13%),4 (5.63%),8 (11.27%),2 (2.82%),34 (47.89%),5 (7.04%)
32,ENSINO,62,2 (3.23%),0 (0.0%),21 (33.87%),7 (11.29%),7 (11.29%),1 (1.61%),20 (32.26%),4 (6.45%)


## Status de Projetos

In [69]:
analise_area_conhecimento_status = projetos_df.groupby("Área de Conhecimento").agg(
    projetos_inscritos=("ID do Projeto", "count"),
    projetos_concluidos=("concluido", "sum"),
    projetos_em_edicao=("em_edicao", "sum"),
    projetos_pre_aprovados=("pre_aprovado", "sum"),
    projetos_inativados=("inativado", "sum"),
    projetos_nao_enviados=("nao_enviado", "sum"),
    projetos_enviados=("enviado", "sum"),
    projetos_aprovados=("Aprovado", "sum")
).reset_index().rename(columns={
    "projetos_inscritos": "Projetos Inscritos",
    "projetos_concluidos": "Projetos Concluídos",
    "projetos_em_edicao": "Projetos em Edição",
    "projetos_pre_aprovados": "Projetos Pré-Aprovados",
    "projetos_inativados": "Projetos Inativados",
    "projetos_nao_enviados": "Projetos Não Enviados",
    "projetos_enviados": "Projetos Enviados",
    "projetos_aprovados": "Projetos Aprovados"
})

analise_area_conhecimento_status['Projetos Concluídos'] = (
    analise_area_conhecimento_status['Projetos Concluídos'].astype(str) + 
    ' (' + (analise_area_conhecimento_status['Projetos Concluídos'] / analise_area_conhecimento_status['Projetos Inscritos'] * 100).round(2).astype(str) + '%)'
)

analise_area_conhecimento_status['Projetos em Edição'] = (
    analise_area_conhecimento_status['Projetos em Edição'].astype(str) + 
    ' (' + (analise_area_conhecimento_status['Projetos em Edição'] / analise_area_conhecimento_status['Projetos Inscritos'] * 100).round(2).astype(str) + '%)'
)

analise_area_conhecimento_status['Projetos Pré-Aprovados'] = (
    analise_area_conhecimento_status['Projetos Pré-Aprovados'].astype(str) +
    ' (' + (analise_area_conhecimento_status['Projetos Pré-Aprovados'] / analise_area_conhecimento_status['Projetos Inscritos'] * 100).round(2).astype(str) + '%)'
)

analise_area_conhecimento_status['Projetos Inativados'] = (
    analise_area_conhecimento_status['Projetos Inativados'].astype(str) +
    ' (' + (analise_area_conhecimento_status['Projetos Inativados'] / analise_area_conhecimento_status['Projetos Inscritos'] * 100).round(2).astype(str) + '%)'
)

analise_area_conhecimento_status['Projetos Não Enviados'] = (
    analise_area_conhecimento_status['Projetos Não Enviados'].astype(str) +
    ' (' + (analise_area_conhecimento_status['Projetos Não Enviados'] / analise_area_conhecimento_status['Projetos Inscritos'] * 100).round(2).astype(str) + '%)'
)

analise_area_conhecimento_status['Projetos Enviados'] = (
    analise_area_conhecimento_status['Projetos Enviados'].astype(str) +
    ' (' + (analise_area_conhecimento_status['Projetos Enviados'] / analise_area_conhecimento_status['Projetos Inscritos'] * 100).round(2).astype(str) + '%)' 
)

analise_area_conhecimento_status['Projetos Aprovados'] = (
    analise_area_conhecimento_status['Projetos Aprovados'].astype(str) +
    ' (' + (analise_area_conhecimento_status['Projetos Aprovados'] / analise_area_conhecimento_status['Projetos Inscritos'] * 100).round(2).astype(str) + '%)'
)

display(analise_area_conhecimento_status)

,Área de Conhecimento,Projetos Inscritos,Projetos Concluídos,Projetos em Edição,Projetos Pré-Aprovados,Projetos Inativados,Projetos Não Enviados,Projetos Enviados,Projetos Aprovados
0,ADMINISTRAÇÃO,51,13 (25.49%),0 (0.0%),38 (74.51%),3 (5.88%),12 (23.53%),39 (76.47%),28 (54.9%)
1,AGRONOMIA,228,81 (35.53%),4 (1.75%),205 (89.91%),12 (5.26%),16 (7.02%),208 (91.23%),119 (52.19%)
2,ANTROPOLOGIA,6,2 (33.33%),0 (0.0%),5 (83.33%),1 (16.67%),0 (0.0%),6 (100.0%),3 (50.0%)
3,ARQUITETURA E URBANISMO,21,10 (47.62%),0 (0.0%),18 (85.71%),1 (4.76%),3 (14.29%),18 (85.71%),14 (66.67%)
4,ARTES,49,13 (26.53%),1 (2.04%),35 (71.43%),3 (6.12%),9 (18.37%),38 (77.55%),22 (44.9%)
5,ASTRONOMIA,16,4 (25.0%),0 (0.0%),12 (75.0%),0 (0.0%),4 (25.0%),12 (75.0%),8 (50.0%)
6,BIOFÍSICA,1,0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),1 (100.0%),0 (0.0%),0 (0.0%)
7,BIOLOGIA GERAL,24,7 (29.17%),0 (0.0%),22 (91.67%),0 (0.0%),1 (4.17%),23 (95.83%),17 (70.83%)
8,BIOQUÍMICA,16,1 (6.25%),0 (0.0%),10 (62.5%),2 (12.5%),2 (12.5%),14 (87.5%),4 (25.0%)
9,BIOTECNOLOGIA,35,12 (34.29%),1 (2.86%),29 (82.86%),4 (11.43%),4 (11.43%),30 (85.71%),17 (48.57%)


# Pesquisadores

## Pesquisadores por Ano

In [ ]:
analise_pesquisadores_ano = membros_projetos_df.groupby("Ano de Início da Execução").agg(
    total_pesquisadores=("Nome do Membro", "count"),
    bolsistas=("Vínculo com Projeto", lambda x: (x == "Bolsista").sum()),
    voluntarios=("Vínculo com Projeto", lambda x: (x == "Voluntário").sum()),
).reset_index().rename(columns={"total_pesquisadores": "Total de Pesquisadores", "bolsistas": "Bolsistas", "voluntarios": "Voluntários"})

analise_pesquisadores_ano['Bolsistas'] = (
    analise_pesquisadores_ano['Bolsistas'].astype(str) +
    ' (' + (analise_pesquisadores_ano['Bolsistas'] / analise_pesquisadores_ano['Total de Pesquisadores'] * 100).round(2).astype(str) + '%)'
)

analise_pesquisadores_ano['Voluntários'] = (
    analise_pesquisadores_ano['Voluntários'].astype(str) +
    ' (' + (analise_pesquisadores_ano['Voluntários'] / analise_pesquisadores_ano['Total de Pesquisadores'] * 100).round(2).astype(str) + '%)'
)

display(analise_pesquisadores_ano)

,Ano de Início da Execução,Total de Pesquisadores,Bolsistas,Voluntários
0,2014.0,1,0 (0.0%),1 (100.0%)
1,2022.0,20,12 (60.0%),8 (40.0%)
2,2023.0,2029,615 (30.31%),1414 (69.69%)
3,2024.0,2131,648 (30.41%),1483 (69.59%)
4,2025.0,3018,873 (28.93%),2145 (71.07%)
5,2026.0,377,42 (11.14%),335 (88.86%)


## Pesquisadores por Edital

In [74]:
analise_pesquisadores_edital = membros_projetos_df.groupby("Categoria do Membro").agg(
    total_projetos=("ID do Projeto", "count"),
    total_externo=("Edital", lambda x: (x == "FOMENTO EXTERNO").sum()),
    total_interno=("Edital", lambda x: (x == "FOMENTO INTERNO").sum()),
    total_pibic=("Edital", lambda x: (x == "PIBIC").sum()),
    total_pibic_af=("Edital", lambda x: (x == "PIBIC AF").sum()),
    total_pibic_jr=("Edital", lambda x: (x == "PIBIC JR").sum()),
    total_pibic_jr_af=("Edital", lambda x: (x == "PIBIC JR AF").sum()),
    total_pictv=("Edital", lambda x: (x == "PICTV").sum()),
    total_pibiti=("Edital", lambda x: (x == "PIBITI").sum())
).reset_index().rename(columns={
    "total_projetos": "Total de Projetos",
    "total_externo": "Fomento Externo",
    "total_interno": "Fomento Interno",
    "total_pibic": "PIBIC",
    "total_pibic_af": "PIBIC AF",
    "total_pibic_jr": "PIBIC JR",
    "total_pibic_jr_af": "PIBIC JR AF",
    "total_pictv": "PICTV",
    "total_pibiti": "PIBITI"
})
analise_pesquisadores_edital = analise_pesquisadores_edital.sort_values(by="Total de Projetos", ascending=False)

analise_pesquisadores_edital['Fomento Externo'] = (
    analise_pesquisadores_edital['Fomento Externo'].astype(str) +
    ' (' + (analise_pesquisadores_edital['Fomento Externo'] / analise_pesquisadores_edital['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

analise_pesquisadores_edital['Fomento Interno'] = (
    analise_pesquisadores_edital['Fomento Interno'].astype(str) +
    ' (' + (analise_pesquisadores_edital['Fomento Interno'] / analise_pesquisadores_edital['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

analise_pesquisadores_edital['PIBIC'] = (
    analise_pesquisadores_edital['PIBIC'].astype(str) + 
    ' (' + (analise_pesquisadores_edital['PIBIC'] / analise_pesquisadores_edital['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

analise_pesquisadores_edital['PIBIC AF'] = (
    analise_pesquisadores_edital['PIBIC AF'].astype(str) +
    ' (' + (analise_pesquisadores_edital['PIBIC AF'] / analise_pesquisadores_edital['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

analise_pesquisadores_edital['PIBIC JR'] = (
    analise_pesquisadores_edital['PIBIC JR'].astype(str) +
    ' (' + (analise_pesquisadores_edital['PIBIC JR'] / analise_pesquisadores_edital['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

analise_pesquisadores_edital['PIBIC JR AF'] = (
    analise_pesquisadores_edital['PIBIC JR AF'].astype(str) +
    ' (' + (analise_pesquisadores_edital['PIBIC JR AF'] / analise_pesquisadores_edital['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

analise_pesquisadores_edital['PICTV'] = (
    analise_pesquisadores_edital['PICTV'].astype(str) +
    ' (' + (analise_pesquisadores_edital['PICTV'] / analise_pesquisadores_edital['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

analise_pesquisadores_edital['PIBITI'] = (
    analise_pesquisadores_edital['PIBITI'].astype(str) +
    ' (' + (analise_pesquisadores_edital['PIBITI'] / analise_pesquisadores_edital['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

display(analise_pesquisadores_edital)

,Categoria do Membro,Total de Projetos,Fomento Externo,Fomento Interno,PIBIC,PIBIC AF,PIBIC JR,PIBIC JR AF,PICTV,PIBITI
1,Docente,4194,596 (14.21%),101 (2.41%),1114 (26.56%),321 (7.65%),514 (12.26%),134 (3.2%),860 (20.51%),554 (13.21%)
0,Discente,3042,237 (7.79%),71 (2.33%),594 (19.53%),96 (3.16%),330 (10.85%),90 (2.96%),1336 (43.92%),288 (9.47%)
3,TAE,242,77 (31.82%),21 (8.68%),38 (15.7%),12 (4.96%),17 (7.02%),12 (4.96%),53 (21.9%),12 (4.96%)
2,Não Informado,98,45 (45.92%),2 (2.04%),4 (4.08%),0 (0.0%),17 (17.35%),5 (5.1%),23 (23.47%),2 (2.04%)


## Pesquisadores por Área de Conhecimento

In [75]:
analise_pesquisadores_area_conhecimento = membros_projetos_df.groupby("Área de Conhecimento").agg(
    total_projetos=("ID do Projeto", "count"),
    discente=("Categoria do Membro", lambda x: (x == "Discente").sum()),
    docente=("Categoria do Membro", lambda x: (x == "Docente").sum()),
    tae=("Categoria do Membro", lambda x: (x == "TAE").sum()),
    outros=("Categoria do Membro", lambda x: (x == "Não Informado").sum())
).reset_index().rename(columns={
    "total_projetos": "Total de Projetos",
    "discente": "Discente",
    "docente": "Docente",
    "tae": "TAE",
    "outros": "Não Informado"
}).sort_values(by="Total de Projetos", ascending=False)

analise_pesquisadores_area_conhecimento['Discente'] = (
    analise_pesquisadores_area_conhecimento['Discente'].astype(str) +
    ' (' + (analise_pesquisadores_area_conhecimento['Discente'] / analise_pesquisadores_area_conhecimento['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

analise_pesquisadores_area_conhecimento['Docente'] = (
    analise_pesquisadores_area_conhecimento['Docente'].astype(str) +
    ' (' + (analise_pesquisadores_area_conhecimento['Docente'] / analise_pesquisadores_area_conhecimento['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

analise_pesquisadores_area_conhecimento['TAE'] = (
    analise_pesquisadores_area_conhecimento['TAE'].astype(str) +
    ' (' + (analise_pesquisadores_area_conhecimento['TAE'] / analise_pesquisadores_area_conhecimento['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

analise_pesquisadores_area_conhecimento['Não Informado'] = (
    analise_pesquisadores_area_conhecimento['Não Informado'].astype(str) +
    ' (' + (analise_pesquisadores_area_conhecimento['Não Informado'] / analise_pesquisadores_area_conhecimento['Total de Projetos'] * 100).round(2).astype(str) + '%)'
)

display(analise_pesquisadores_area_conhecimento)

,Área de Conhecimento,Total de Projetos,Discente,Docente,TAE,Não Informado
11,CIÊNCIA DA COMPUTAÇÃO,1524,588 (38.58%),815 (53.48%),82 (5.38%),39 (2.56%)
30,ENGENHARIA ELÉTRICA,663,197 (29.71%),444 (66.97%),13 (1.96%),9 (1.36%)
1,AGRONOMIA,453,179 (39.51%),246 (54.3%),21 (4.64%),7 (1.55%)
34,ENGENHARIA SANITÁRIA,398,174 (43.72%),201 (50.5%),17 (4.27%),6 (1.51%)
20,EDUCAÇÃO,388,168 (43.3%),210 (54.12%),10 (2.58%),0 (0.0%)
57,QUÍMICA,387,155 (40.05%),214 (55.3%),15 (3.88%),3 (0.78%)
13,CIÊNCIA E TECNOLOGIA DE ALIMENTOS,356,134 (37.64%),213 (59.83%),7 (1.97%),2 (0.56%)
44,INTERDISCIPLINAR,299,123 (41.14%),173 (57.86%),2 (0.67%),1 (0.33%)
35,ENSINO,242,107 (44.21%),130 (53.72%),1 (0.41%),4 (1.65%)
21,EDUCAÇÃO FÍSICA,232,90 (38.79%),142 (61.21%),0 (0.0%),0 (0.0%)


# Grupos de Pesquisa

In [78]:
total_grupos_de_pesquisa = grupos_pesquisa_df["Nome do Grupo de Pesquisa"].nunique()
print(f"Total de Grupos de Pesquisa: {total_grupos_de_pesquisa}")

total_projetos_grupo_pesquisa = grupos_pesquisa_df["Total de Projetos Aprovados"].sum()
print(f"Total de Projetos Aprovados em Grupos de Pesquisa: {total_projetos_grupo_pesquisa}")

Total de Grupos de Pesquisa: 174
Total de Projetos Aprovados em Grupos de Pesquisa: 1695


# Produção Técnica

In [79]:
total_producoes_tecnicas = producoes_tecnicas_df.shape[0]
print(f"Total de Produções Técnicas: {total_producoes_tecnicas}")

analise_producoes_tecnicas_por_tipo = producoes_tecnicas_df.loc[:, ["Tipo de Publicação", "ID"]]
analise_producoes_tecnicas_por_tipo = analise_producoes_tecnicas_por_tipo.groupby("Tipo de Publicação").count().reset_index()
analise_producoes_tecnicas_por_tipo = analise_producoes_tecnicas_por_tipo.rename(columns={
    "ID": "Total de Produções Técnicas"
})
analise_producoes_tecnicas_por_tipo = analise_producoes_tecnicas_por_tipo.sort_values(by="Total de Produções Técnicas", ascending=False)
analise_producoes_tecnicas_por_tipo["Total de Produções Técnicas"] = (
    analise_producoes_tecnicas_por_tipo["Total de Produções Técnicas"].astype(str)
    + " ("
    + (
        analise_producoes_tecnicas_por_tipo["Total de Produções Técnicas"].astype(int)
        / total_producoes_tecnicas * 100
    ).round(2).astype(str)
    + "%)"
)
display(analise_producoes_tecnicas_por_tipo)


Total de Produções Técnicas: 83658


,Tipo de Publicação,Total de Produções Técnicas
0,Apresentação De Trabalho,36337 (43.44%)
8,Organização De Evento,16656 (19.91%)
2,Curso De Curta Duração Ministrado,10608 (12.68%)
16,Trabalho Tecnico,8327 (9.95%)
13,Programa De Radio Ou Tv,3057 (3.65%)
9,Outra Produção Tecnica,2427 (2.9%)
3,Desenvolvimento De Material Didatico Ou Instrucional,2320 (2.77%)
15,Software,1044 (1.25%)
14,Relatorio De Pesquisa,747 (0.89%)
10,Patente,554 (0.66%)


# Orientações

In [86]:
orientacoes_andamento = orientacoes_df[orientacoes_df["Situação"] == "Em Andamento"].shape[0]
orientacoes_concluidas = orientacoes_df[orientacoes_df["Situação"] == "Concluida"].shape[0]
print(f"Total de Orientações em Andamento: {orientacoes_andamento}" + f" ({orientacoes_andamento / orientacoes_df.shape[0] * 100:.2f}%)")
print(f"Total de Orientações Concluídas: {orientacoes_concluidas}" + f" ({orientacoes_concluidas / orientacoes_df.shape[0] * 100:.2f}%)")

orientacoes_com_bolsa = orientacoes_df[orientacoes_df["flag_bolsa"] == True].shape[0]
orientacoes_sem_bolsa = orientacoes_df[orientacoes_df["flag_bolsa"] == False].shape[0]
print(f"Total de Orientações com Bolsa: {orientacoes_com_bolsa}" + f" ({orientacoes_com_bolsa / orientacoes_df.shape[0] * 100:.2f}%)")
print(f"Total de Orientações sem Bolsa: {orientacoes_sem_bolsa}" + f" ({orientacoes_sem_bolsa / orientacoes_df.shape[0] * 100:.2f}%)")

Total de Orientações em Andamento: 3603 (7.57%)
Total de Orientações Concluídas: 43964 (92.43%)
Total de Orientações com Bolsa: 9281 (19.51%)
Total de Orientações sem Bolsa: 38286 (80.49%)


## Por Ano

In [88]:
analise_orientacoes_por_ano = orientacoes_df.groupby("ano").agg(
    total_orientacoes=("id", "count"),
    orientacoes_com_bolsa=("flag_bolsa", lambda x: (x == True).sum()),
    orientacoes_sem_bolsa=("flag_bolsa", lambda x: (x == False).sum())
).reset_index().rename(columns={
    "total_orientacoes": "Total de Orientações",
    "orientacoes_com_bolsa": "Orientações com Bolsa",
    "orientacoes_sem_bolsa": "Orientações sem Bolsa"
}).sort_values(by="ano")

analise_orientacoes_por_ano['Orientações com Bolsa'] = (
    analise_orientacoes_por_ano['Orientações com Bolsa'].astype(str) +
    ' (' + (analise_orientacoes_por_ano['Orientações com Bolsa'] / analise_orientacoes_por_ano['Total de Orientações'] * 100).round(2).astype(str) + '%)'
)

analise_orientacoes_por_ano['Orientações sem Bolsa'] = (
    analise_orientacoes_por_ano['Orientações sem Bolsa'].astype(str) + 
    ' (' + (analise_orientacoes_por_ano['Orientações sem Bolsa'] / analise_orientacoes_por_ano['Total de Orientações'] * 100).round(2).astype(str) + '%)'
)

display(analise_orientacoes_por_ano)

,ano,Total de Orientações,Orientações com Bolsa,Orientações sem Bolsa
0,2004,2,0 (0.0%),2 (100.0%)
1,2005,2,0 (0.0%),2 (100.0%)
2,2006,11,1 (9.09%),10 (90.91%)
3,2007,24,5 (20.83%),19 (79.17%)
4,2008,27,11 (40.74%),16 (59.26%)
5,2009,12,3 (25.0%),9 (75.0%)
6,2010,38,21 (55.26%),17 (44.74%)
7,2011,64,25 (39.06%),39 (60.94%)
8,2012,100,49 (49.0%),51 (51.0%)
9,2013,116,50 (43.1%),66 (56.9%)
